In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
user_credential = user_secrets.get_gcloud_credential()
user_secrets.set_tensorflow_credential(user_credential)
user_secrets.set_gcloud_credentials(project="mobilewaft")


In [ ]:
from __future__ import annotations

import gzip
import io
import json
import random
import concurrent.futures
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional
import os
import time
import threading

import numpy as np
from google.cloud import storage
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseUpload
from google.oauth2 import service_account
import google.auth
import dotenv
dotenv.load_dotenv()


# ─────────────────────────────────────────────
# Config
# ─────────────────────────────────────────────

@dataclass
class SamplerConfig:
    bucket_name: str = "gresearch"
    prefix: str = "sanpo_dataset/v0/sanpo-real/"
    max_sessions: int = 500
    batch_size: int = 50
    max_sample: int = 100        # frame triples per session per camera
    stride: int = 15             # lấy 1 frame mỗi N frames để tránh near-duplicate
    seed: int = 42
    cameras: List[str] = field(default_factory=lambda: ["head", "chest"])
    require_all_modalities: bool = True
    size_limit_gb: float = 100.0


# ─────────────────────────────────────────────
# Sampler
# ─────────────────────────────────────────────

class SanpoDepthPairSampler:
    def __init__(self, config: SamplerConfig):
        self.cfg = config
        self.client = storage.Client.create_anonymous_client()
        self.rng = random.Random(config.seed)

    def _gs_uri(self, object_name: str) -> str:
        return f"gs://{self.cfg.bucket_name}/{object_name}"

    def _session_prefix(self, session_id: str) -> str:
        return f"{self.cfg.prefix.rstrip('/')}/{session_id}/"

    def _calib_url(self, session_id: str) -> str:
        return self._gs_uri(f"{self._session_prefix(session_id)}description.json")

    def _list_prefixes(self, prefix: str) -> List[str]:
        it = self.client.list_blobs(
            self.cfg.bucket_name, prefix=prefix, delimiter="/"
        )
        prefixes = set()
        for page in it.pages:
            prefixes.update(page.prefixes)
        return sorted(prefixes)

    def _list_files_flat(self, prefix: str) -> List[str]:
        it = self.client.list_blobs(
            self.cfg.bucket_name, prefix=prefix, delimiter="/"
        )
        out = []
        for page in it.pages:
            for blob in page:
                base = blob.name.rsplit("/", 1)[-1]
                if base and not base.endswith("$folder$"):
                    out.append(blob.name)
        return sorted(out)

    def _normalize_id(self, path: str) -> str:
        name = path.rsplit("/", 1)[-1]
        if name.endswith(".float16.gz"): return name[:-11]
        return name.rsplit(".", 1)[0] if "." in name else name

    def _frame_map(self, session_id: str, rel_prefix: str) -> Dict[str, str]:
        prefix = f"{self._session_prefix(session_id)}{rel_prefix.strip('/')}/"
        files = self._list_files_flat(prefix)
        return {self._normalize_id(f): f for f in files}

    def _sample_camera(self, left_map, right_map, depth_map) -> List[dict]:
        sets = [set(m.keys()) for m in (left_map, right_map, depth_map) if m]
        if not sets: return []
        valid_ids = sorted(
            set.intersection(*sets) if self.cfg.require_all_modalities
            else set.union(*sets)
        )
        if not valid_ids: return []
        strided = valid_ids[:: max(1, self.cfg.stride)]
        chosen = self.rng.sample(strided, k=min(self.cfg.max_sample, len(strided)))
        return [
            {
                "id": fid,
                "left":  self._gs_uri(left_map[fid])  if fid in left_map  else None,
                "right": self._gs_uri(right_map[fid]) if fid in right_map else None,
                "depth": self._gs_uri(depth_map[fid]) if fid in depth_map else None,
            }
            for fid in chosen
        ]

    def dry_run(self, ignored_sessions: Optional[List[str]] = None, num_sessions_to_select: Optional[int] = None) -> List[dict]:
        """
        Args:
            ignored_sessions: List session_id gốc (hash) đã processed.
            num_sessions_to_select: Số lượng session cần select cho batch này.
        """
        all_prefixes = self._list_prefixes(self.cfg.prefix.rstrip("/") + "/")
        session_ids = [p.rstrip("/").split("/")[-1] for p in all_prefixes]
        if not session_ids: return []

        ignored_set = set(ignored_sessions or [])
        available = [s for s in session_ids if s not in ignored_set]
        if not available:
            print("[sampler] Tất cả session đã được download.")
            return []

        k = num_sessions_to_select or self.cfg.batch_size
        chosen = self.rng.sample(available, k=min(k, len(available)))
        CAM_DIRS = {"head": "camera_head", "chest": "camera_chest"}
        results = []

        for sid in chosen:
            entry = {
                "session_name": sid,
                "calib_url": self._calib_url(sid),
            }
            for cam in self.cfg.cameras:
                cdir = CAM_DIRS[cam]
                entry[cam] = self._sample_camera(
                    self._frame_map(sid, f"{cdir}/left/video_frames"),
                    self._frame_map(sid, f"{cdir}/right/video_frames"),
                    self._frame_map(sid, f"{cdir}/left/depth_maps"),
                )
            results.append(entry)

        total = sum(len(r.get(c, [])) for r in results for c in self.cfg.cameras)
        print(f"[sampler] Chosen {len(results)} sessions | {total} frame triples")
        return results


# ─────────────────────────────────────────────
# Helpers: Calib + Depth
# ─────────────────────────────────────────────

def _extract_calib(desc: dict, camera: str) -> dict:
    cam_key = "camera_head" if camera == "head" else "camera_chest"
    locations = desc.get("session_camera_location", [])
    details   = desc.get("session_camera_details", [])
    cam_detail = next(
        (details[i] for i, loc in enumerate(locations) if loc == cam_key and i < len(details)),
        details[1 if camera == "head" else 0] if details else None,
    )
    if not cam_detail: return {}
    lp = cam_detail["left_camera_params"]
    return {
        "camera":           cam_key,
        "focal_length_px":  lp["fx"],
        "baseline_m":       round(abs(cam_detail["stereo_transform"]["coeff"][3]) / 1000.0, 8),
        "cx":               lp["cx"],
        "cy":               lp["cy"],
        "image_width":      lp["image_width"],
        "image_height":     lp["image_height"],
        "fps":              cam_detail.get("fps"),
        "model":            cam_detail.get("model"),
    }


def _decode_float16_gz(raw: bytes) -> np.ndarray:
    with gzip.open(io.BytesIO(raw), "rb") as f:
        data = np.frombuffer(f.read(), dtype=np.float16)
    h = int(data[0])
    w = int(data[1])
    return data[2:].reshape(h, w).astype(np.float32)


# ─────────────────────────────────────────────
# Session Index Helpers
# ─────────────────────────────────────────────

def _load_session_index(output_dir: str) -> Dict[str, str]:
    index_file = Path(output_dir) / "session_index.json"
    if index_file.exists():
        with open(index_file, "r") as f:
            return json.load(f)
    return {}


def _save_session_index(output_dir: str, index: Dict[str, str]) -> None:
    index_file = Path(output_dir) / "session_index.json"
    index_file.parent.mkdir(parents=True, exist_ok=True)
    with open(index_file, "w") as f:
        json.dump(index, f, indent=2)


def _next_session_number(index: Dict[str, str]) -> int:
    if not index:
        return 1
    existing = [int(v.split("_")[1]) for v in index.values() if "_" in v]
    return max(existing) + 1 if existing else 1


def _load_processed_sessions(output_dir: str) -> List[str]:
    p_file = Path(output_dir) / "processed_sessions.json"
    if p_file.exists():
        with open(p_file, "r") as f:
            return json.load(f)
    return []


def _save_processed_sessions(output_dir: str, processed: List[str]) -> None:
    p_file = Path(output_dir) / "processed_sessions.json"
    p_file.parent.mkdir(parents=True, exist_ok=True)
    with open(p_file, "w") as f:
        json.dump(processed, f, indent=2)


# ─────────────────────────────────────────────
# Safe Counter for Progress Tracking
# ─────────────────────────────────────────────

class SafeCounter:
    def __init__(self):
        self.value = 0
        self.lock = threading.Lock()
    def add(self, n):
        with self.lock:
            self.value += n
    def get(self):
        with self.lock:
            return self.value


# ─────────────────────────────────────────────
# Google Drive Authentication Service
# ─────────────────────────────────────────────

def get_gdrive_service():
    # 1. Try Service Account JSON secret first
    try:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        sa_json = user_secrets.get_secret("GD_SERVICE_ACCOUNT_JSON")
        if sa_json:
            print("Using Google Drive Service Account credentials...")
            sa_info = json.loads(sa_json)
            creds = service_account.Credentials.from_service_account_info(
                sa_info,
                scopes=["https://www.googleapis.com/auth/drive"]
            )
            return build('drive', 'v3', credentials=creds)
    except Exception as e:
        print(f"Service Account authentication skipped/failed: {e}")

    # 2. Try converting standard Kaggle Google Cloud User credentials (returned as JSON string)
    try:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        user_credential = user_secrets.get_gcloud_credential()
        if user_credential:
            print("Using standard Kaggle Google Cloud User credentials...")
            info = json.loads(user_credential)
            if info.get("type") == "service_account" or "private_key" in info:
                from google.oauth2 import service_account
                creds = service_account.Credentials.from_service_account_info(
                    info,
                    scopes=["https://www.googleapis.com/auth/drive"]
                )
            else:
                from google.oauth2.credentials import Credentials
                creds = Credentials.from_authorized_user_info(
                    info,
                    scopes=["https://www.googleapis.com/auth/drive"]
                )
            return build('drive', 'v3', credentials=creds)
    except Exception as e:
        print(f"Kaggle Google Cloud User credentials skipped/failed: {e}")

    # 3. Try standard Google Auth default credentials (linked Kaggle Google Cloud SDK user login)
    try:
        print("Using standard Google Auth default credentials...")
        creds, project = google.auth.default()
        try:
            creds = creds.with_scopes(["https://www.googleapis.com/auth/drive"])
        except Exception as scope_err:
            print(f"Note: Could not explicitly add drive scope: {scope_err}")
        return build('drive', 'v3', credentials=creds)
    except Exception as e:
        print(f"Google Auth default credentials failed: {e}")
        
    raise ValueError("Google Drive credentials not configured. Please link your Google account in Kaggle Add-ons or define GD_SERVICE_ACCOUNT_JSON.")


def create_gdrive_folder(name: str, parent_id: str, drive_service) -> str:
    # Idempotent: check if folder already exists (supports Team Drives / Shared Drives)
    query = f"name = '{name}' and '{parent_id}' in parents and mimeType = 'application/vnd.google-apps.folder' and trashed = false"
    results = drive_service.files().list(
        q=query, 
        spaces='drive', 
        fields='files(id, name)',
        supportsAllDrives=True,
        includeItemsFromAllDrives=True
    ).execute()
    files = results.get('files', [])
    if files:
        return files[0]['id']
        
    file_metadata = {
        'name': name,
        'mimeType': 'application/vnd.google-apps.folder',
        'parents': [parent_id]
    }
    folder = drive_service.files().create(
        body=file_metadata, 
        fields='id',
        supportsAllDrives=True
    ).execute()
    return folder.get('id')


In [ ]:
def run_simultaneous_download_upload(
    results: List[dict],
    parent_folder_id: str,
    drive_service,
    bucket_name: str = "gresearch",
    output_dir: str = "/kaggle/working/sanpo_real",
    max_workers: int = 8,
    cameras: Optional[List[str]] = None,
    counter: Optional[SafeCounter] = None,
) -> None:
    """
    Tải từ GCS và tải lên Google Drive song song, hoàn toàn in-memory thông qua BytesIO
    để không tiêu tốn dung lượng ổ đĩa của Kaggle.
    """
    if cameras is None:
        cameras = [c for c in ["head", "chest"] if any(c in r for r in results)]

    client = storage.Client.create_anonymous_client()

    def fetch(uri: str) -> bytes:
        obj = uri[len(f"gs://{bucket_name}/"):]
        return client.bucket(bucket_name).blob(obj).download_as_bytes()

    # ── Assign session folder mapping ──────────────────────────────────────────
    index = _load_session_index(output_dir)
    next_num = _next_session_number(index)

    for s in results:
        sid = s["session_name"]
        for cam in cameras:
            if not s.get(cam):
                continue
            key = f"{sid}_{cam}"
            if key not in index:
                index[key] = f"session_{next_num:04d}"
                next_num += 1
    _save_session_index(output_dir, index)

    # ── Pre-create Google Drive folder structure ───────────────────────────────
    gdrive_map = {}
    print("[gdrive] Pre-creating folder layout on Google Drive...")
    for s in results:
        sid = s["session_name"]
        desc = None
        if s.get("calib_url"):
            try:
                desc = json.loads(fetch(s["calib_url"]))
            except Exception as e:
                print(f"[WARN] calib fetch failed {sid}: {e}")

        for cam in cameras:
            if not s.get(cam):
                continue
            key = f"{sid}_{cam}"
            folder_name = index[key]
            
            # Create session folder
            sess_fid = create_gdrive_folder(folder_name, parent_folder_id, drive_service)
            
            # Create subfolders inside session folder
            left_fid = create_gdrive_folder("left", sess_fid, drive_service)
            right_fid = create_gdrive_folder("right", sess_fid, drive_service)
            left_depth_fid = create_gdrive_folder("depth_ml", sess_fid, drive_service)
            
            gdrive_map[key] = {
                "session": sess_fid,
                "left": left_fid,
                "right": right_fid,
                "depth_ml": left_depth_fid
            }
            
            # Upload calib.json to Google Drive
            if desc:
                try:
                    calib = _extract_calib(desc, cam)
                    calib_bytes = json.dumps(calib, indent=2).encode('utf-8')
                    media = MediaIoBaseUpload(io.BytesIO(calib_bytes), mimetype='application/json')
                    file_metadata = {'name': 'calib.json', 'parents': [sess_fid]}
                    drive_service.files().create(
                        body=file_metadata, 
                        media_body=media, 
                        fields='id',
                        supportsAllDrives=True
                    ).execute()
                except Exception as e:
                    print(f"[WARN] Failed to upload calib.json for {folder_name}: {e}")

    # ── Build task list ────────────────────────────────────────────────────────
    tasks = []
    for s in results:
        sid = s["session_name"]
        for cam in cameras:
            if not s.get(cam):
                continue
            key = f"{sid}_{cam}"
            folders = gdrive_map[key]
            for frame in s.get(cam, []):
                tasks.append((frame, folders))

    total = len(tasks)
    print(f"[pipeline] Starting upload for {total} frames using {max_workers} threads...")

    # ── Parallel fetch & upload ────────────────────────────────────────────────
    done = errors = 0

    def process(task):
        nonlocal done, errors
        frame, folders = task
        fid = frame["id"]
        uploaded_bytes = 0
        try:
            # 1. Left image
            if frame.get("left"):
                left_data = fetch(frame["left"])
                media = MediaIoBaseUpload(io.BytesIO(left_data), mimetype='image/png')
                file_metadata = {'name': f"{fid}.png", 'parents': [folders["left"]]}
                drive_service.files().create(
                    body=file_metadata, 
                    media_body=media, 
                    fields='id',
                    supportsAllDrives=True
                ).execute()
                uploaded_bytes += len(left_data)

            # 2. Right image
            if frame.get("right"):
                right_data = fetch(frame["right"])
                media = MediaIoBaseUpload(io.BytesIO(right_data), mimetype='image/png')
                file_metadata = {'name': f"{fid}.png", 'parents': [folders["right"]]}
                drive_service.files().create(
                    body=file_metadata, 
                    media_body=media, 
                    fields='id',
                    supportsAllDrives=True
                ).execute()
                uploaded_bytes += len(right_data)

            # 3. Depth map
            if frame.get("depth"):
                depth_data = fetch(frame["depth"])
                arr = _decode_float16_gz(depth_data)
                
                # Save array to bytes buffer
                buf = io.BytesIO()
                np.save(buf, arr)
                buf.seek(0)
                
                media = MediaIoBaseUpload(buf, mimetype='application/octet-stream')
                file_metadata = {'name': f"{fid}.npy", 'parents': [folders["depth_ml"]]}
                drive_service.files().create(
                    body=file_metadata, 
                    media_body=media, 
                    fields='id',
                    supportsAllDrives=True
                ).execute()
                uploaded_bytes += len(depth_data)

            if counter:
                counter.add(uploaded_bytes)
        except Exception as e:
            print(f"[ERROR] Pipeline upload failed for frame {fid}: {e}")
            raise e

    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as pool:
        futures = {pool.submit(process, t): t for t in tasks}
        for fut in concurrent.futures.as_completed(futures):
            done += 1
            if fut.exception():
                errors += 1
            if done % 50 == 0 or done == total:
                print(f"  [progress] {done}/{total} files uploaded  errors={errors}")

    print(f"[done] Completed batch upload successfully.")


In [ ]:
import math
from IPython.display import clear_output

# ── Configuration ─────────────────────────────────────────────────────────
MAX_SESSIONS = 500       # Total sessions to sample
BATCH_SIZE = 50          # Batch size to download & upload
SIZE_LIMIT_GB = 100.0    # safety size limit in GB
OUTPUT_DIR = "/kaggle/working/sanpo_real"

# Google Drive destination directory configuration:
# Copy the Folder ID from the URL of your Google Drive destination folder
# Note: If using Service Account credentials, the target folder must be inside a Shared Drive (Team Drive),
# and the Service Account email must be added as a Member with write permissions.
GD_PARENT_FOLDER_ID = "1your_google_drive_folder_id_here"

cfg = SamplerConfig(
    max_sessions=MAX_SESSIONS,
    batch_size=BATCH_SIZE,
    max_sample=100,
    stride=15,
    cameras=["head", "chest"],
    size_limit_gb=SIZE_LIMIT_GB
)

sampler = SanpoDepthPairSampler(cfg)
drive_service = get_gdrive_service()

# Load ignored/processed session list
processed_sessions = _load_processed_sessions(OUTPUT_DIR)
print(f"Already processed: {len(processed_sessions)} sessions.")

total_processed_bytes = 0
batch_idx = len(processed_sessions) // BATCH_SIZE + 1
batch_statuses = [f"Batch {i+1:02d}: COMPLETED (Restored from index)" for i in range(len(processed_sessions) // BATCH_SIZE)]

while len(processed_sessions) < MAX_SESSIONS:
    total_processed_gb = total_processed_bytes / (1024**3)
    if total_processed_gb >= SIZE_LIMIT_GB:
        print(f"Safety limit reached: {total_processed_gb:.2f} GB / {SIZE_LIMIT_GB:.2f} GB. Stopping.")
        break
        
    sessions_remaining = MAX_SESSIONS - len(processed_sessions)
    if sessions_remaining <= 0:
        break
        
    num_sessions_to_select = min(BATCH_SIZE, sessions_remaining)
    
    # Render Dashboard
    clear_output(wait=True)
    print("============================================================")
    print("           SANPO REAL G-DRIVE BATCH PROCESSOR")
    print("============================================================")
    print(f"Configured Max Sessions : {MAX_SESSIONS} (Batch Size: {BATCH_SIZE})")
    print(f"Size Limit              : {SIZE_LIMIT_GB:.2f} GB")
    print(f"Processed Sessions      : {len(processed_sessions)} / {MAX_SESSIONS}")
    print(f"Total Transferred Size  : {total_processed_gb:.4f} GB")
    print("------------------------------------------------------------")
    print("Batch History:")
    for status in batch_statuses:
        print(f"  - {status}")
    print("------------------------------------------------------------")
    print(f"Current Batch {batch_idx:02d} (Selecting {num_sessions_to_select} sessions)...")
    print("============================================================\n")
    
    # 1. Sample sessions for this batch
    results = sampler.dry_run(ignored_sessions=processed_sessions, num_sessions_to_select=num_sessions_to_select)
    if not results:
        print("No more sessions available to process.")
        break
        
    # 2. Run simultaneous download & upload (completely in-memory)
    print(f"\n[Batch {batch_idx:02d}] Executing in-memory pipeline...")
    counter = SafeCounter()
    try:
        run_simultaneous_download_upload(
            results=results,
            parent_folder_id=GD_PARENT_FOLDER_ID,
            drive_service=drive_service,
            bucket_name=cfg.bucket_name,
            output_dir=OUTPUT_DIR,
            max_workers=8,
            counter=counter
        )
        upload_success = True
        batch_bytes = counter.get()
        batch_gb = batch_bytes / (1024**3)
        total_processed_bytes += batch_bytes
    except Exception as e:
        print(f"[ERROR] Pipeline failed for batch {batch_idx:02d}: {e}")
        upload_success = False
        batch_gb = 0.0
        
    # 3. Record progress
    if upload_success:
        for s in results:
            processed_sessions.append(s["session_name"])
        _save_processed_sessions(OUTPUT_DIR, processed_sessions)
        status_msg = f"Batch {batch_idx:02d}: COMPLETED (Transferred {batch_gb:.4f} GB)"
    else:
        status_msg = f"Batch {batch_idx:02d}: FAILED"
        
    batch_statuses.append(status_msg)
    batch_idx += 1

# Final status print
clear_output(wait=True)
print("============================================================")
print("           SANPO REAL BATCH PROCESSOR COMPLETED")
print("============================================================")
print(f"Processed Sessions      : {len(processed_sessions)} / {MAX_SESSIONS}")
print(f"Total Transferred Size  : {total_processed_bytes / (1024**3):.4f} GB")
print("------------------------------------------------------------")
print("Final Batch Statuses:")
for status in batch_statuses:
    print(f"  - {status}")
print("============================================================")


In [ ]:
def visualize_session(
    session_result: dict,
    camera: str = "head",
    n_frames: int = 3,
) -> None:
    """
    Hiển thị n_frames đầu tiên của session: left | right | depth side-by-side
    tải trực tiếp từ Google Cloud Storage về memory để vẽ đồ thị.
    """
    try:
        import matplotlib.pyplot as plt
        import matplotlib.image as mpimg
    except ImportError:
        print("[visualize] Cần cài matplotlib: pip install matplotlib")
        return

    frames = session_result.get(camera, [])
    n = min(n_frames, len(frames))
    if n == 0:
        print(f"[visualize] Không tìm thấy frames cho camera: {camera}")
        return

    client = storage.Client.create_anonymous_client()
    bucket_name = "gresearch"

    def fetch(uri: str) -> bytes:
        obj = uri[len(f"gs://{bucket_name}/"):]
        return client.bucket(bucket_name).blob(obj).download_as_bytes()

    fig, axes = plt.subplots(n, 3, figsize=(15, 5 * n))
    if n == 1:
        axes = [axes]

    fig.suptitle(f"Session: {session_result['session_name']} / Camera: {camera}", fontsize=14, fontweight="bold")

    for i in range(n):
        frame = frames[i]
        fid = frame["id"]
        
        left_bytes = fetch(frame["left"])
        right_bytes = fetch(frame["right"])
        depth_bytes = fetch(frame["depth"])
        
        left_img = mpimg.imread(io.BytesIO(left_bytes), format='png')
        right_img = mpimg.imread(io.BytesIO(right_bytes), format='png')
        depth_arr = _decode_float16_gz(depth_bytes)

        # Left RGB
        axes[i][0].imshow(left_img)
        axes[i][0].set_title(f"Left  [{fid}]", fontsize=9)
        axes[i][0].axis("off")

        # Right RGB
        axes[i][1].imshow(right_img)
        axes[i][1].set_title(f"Right [{fid}]", fontsize=9)
        axes[i][1].axis("off")

        # Depth (colormap plasma)
        vmin = np.percentile(depth_arr[depth_arr > 0], 2)  if np.any(depth_arr > 0) else 0
        vmax = np.percentile(depth_arr[depth_arr > 0], 98) if np.any(depth_arr > 0) else 1
        im = axes[i][2].imshow(depth_arr, cmap="plasma", vmin=vmax, vmax=vmin)
        axes[i][2].set_title(f"Depth (m) [{fid}]", fontsize=9)
        axes[i][2].axis("off")
        plt.colorbar(im, ax=axes[i][2], fraction=0.046, pad=0.04)

    plt.tight_layout()
    plt.show()

print("Visualizer ready. To inspect a chosen session from the dry-run, call: visualize_session(results[0], camera='head')")
